# Step 04. Frenet 기반 Local Planner

> **출처**: PythonRobotics (https://github.com/AtsushiSakai/PythonRobotics)  
> Original author: Atsushi Sakai (@Atsushi_twi) / License: MIT  
>  
> **교육 주관**: Hanyang University, Automotive Intelligence Laboratory (AILAB)

Step 01~03에서 만든 Spline 경로 · Frenet 좌표 · 속도 프로파일을 바탕으로,  
**후보 경로 생성 → 충돌 검사 → 비용 평가 → 최적 선택 → 추종**까지  
Frenet 기반 Local Planner 한 사이클을 **하나의 노트북에서 단계별로** 완성합니다.

> 이전의 후보생성/비용평가/파이프라인 3개 노트북을 통합하고 중복 코드를 제거한 버전입니다.

## 학습 목표

1. d 오프셋 샘플링 → 후보 경로 생성 → 충돌 검사 흐름 이해
2. Safety / Comfort / Reference 비용과 **정규화의 필요성** 이해
3. 가중치 시나리오에 따라 선택 경로가 달라지는 것 확인
4. 최적 경로 선택 + Pure Pursuit 추종 애니메이션
5. 이 방식의 한계 → 다음 단계(Frenet Optimal Trajectory) 동기 부여

## 0. 환경 및 공통 함수

### 이론 배경

Local Planner 한 사이클은 아래 모듈의 **단방향 연결**입니다.

```
d 샘플링 -> 후보 생성 -> 충돌 검사 -> 비용 평가 -> 최적 선택 -> 추종
```

이전에는 `generate_candidate` · `is_collision_free` · 비용 함수가 여러 노트북에 **중복 정의**됐습니다. 여기서는 **한 곳에서만 정의**하고 전체에서 재사용합니다.

In [ ]:
%matplotlib inline
import sys, os, math
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

from utils.cubic_spline import calc_spline_course
from utils.vehicle_model import move, pure_pursuit_steer, VRX, VRY, WB

# ── 공통 환경 (이 노트북 전체에서 재사용) ──────────────────────────
wx = [0.0, 10.0, 20.5, 35.0, 50.0, 60.0]
wy = [0.0, -6.0,  5.0,  6.5,  0.0, -3.0]
rx, ry, ryaw, rk, rs = calc_spline_course(wx, wy, ds=0.5)

obs_x = [23.5, 24.5]   # 경로 좌측(+d) 장애물 클러스터
obs_y = [10.0, 10.0]
ROBOT_RADIUS = 2.0
D_OFFSETS = np.arange(-3.0, 3.5, 0.5)   # 0.5m 간격

# ── 핵심 함수 (한 번만 정의 — 이전엔 3개 노트북에 중복됐던 것) ──────
def generate_candidate(rx, ry, ryaw, d):
    # Reference 경로를 법선 방향으로 d만큼 평행 이동
    cx = [x - math.sin(yaw)*d for x, yaw in zip(rx, ryaw)]
    cy = [y + math.cos(yaw)*d for y, yaw in zip(ry, ryaw)]
    return cx, cy

def is_collision_free(cx, cy, obs_x, obs_y, radius):
    # 경로의 모든 점이 모든 장애물에서 radius 이상 떨어졌는지
    for px, py in zip(cx, cy):
        for ox, oy in zip(obs_x, obs_y):
            if math.hypot(px-ox, py-oy) < radius:
                return False
    return True

def cost_safety(cx, cy, obs_x, obs_y):
    min_d = min(math.hypot(px-ox, py-oy)
                for px, py in zip(cx, cy) for ox, oy in zip(obs_x, obs_y))
    return 1.0 / max(min_d, 1e-6)          # 장애물에 가까울수록 큼

def cost_comfort(cx, cy):
    return sum((math.atan2(cy[i+1]-cy[i], cx[i+1]-cx[i]) -
                math.atan2(cy[i]-cy[i-1], cx[i]-cx[i-1]))**2
               for i in range(1, len(cx)-1))  # 방향 변화 제곱합

def cost_reference(d):
    return d**2                              # 중심선 이탈 제곱

def normalize(values):
    lo, hi = min(values), max(values)
    return [(v-lo)/(hi-lo) if hi > lo else 0.0 for v in values]

print('환경 + 핵심 함수 정의 완료 (함수 중복 제거: 3개 노트북 -> 1곳)')
print(f'  Reference 경로: {rs[-1]:.1f} m, {len(rx)}점')
print(f'  장애물 {len(obs_x)}개, 로봇 반경 {ROBOT_RADIUS} m, d 후보 {len(D_OFFSETS)}개')


## 1. d 오프셋 샘플링

### 이론 배경

Reference 경로에 평행한 후보를 만들기 위해 횡방향 오프셋 `d`를 샘플링합니다.

- `d = 0`: 중심선 유지
- `d > 0`: 좌측, `d < 0`: 우측
- 간격이 좁을수록 정밀하지만 후보 수↑ → 계산량↑

**입력**: 없음(파라미터) / **출력**: d 후보 목록

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
ax.axhline(0, color='blue', lw=3, alpha=0.5)
ax.plot(0, 0, 'o', color='blue', ms=10, zorder=5)
ax.annotate('Reference (d=0)', (0,0), xytext=(0,-0.5), ha='center', fontsize=8, color='blue')
cmap = plt.cm.RdYlGn
for k, d in enumerate(D_OFFSETS):
    color = cmap(0.1 + 0.8*k/(len(D_OFFSETS)-1))
    ax.plot(d, 0, 'o', color=color, ms=13, zorder=4)
    ax.annotate(f'{d:+.1f}', (d,0), xytext=(0,8), textcoords='offset points',
                ha='center', fontsize=8, color=color, fontweight='bold')
ax.annotate('', xy=(3.5,0.6), xytext=(-3.5,0.6), arrowprops=dict(arrowstyle='<->', color='gray', lw=1))
ax.text(0, 0.78, 'lateral offset d [m]  (left = +, right = -)', ha='center', fontsize=9, color='gray')
ax.set_xlim(-4.5,4.5); ax.set_ylim(-1,1.2); ax.axis('off')
ax.set_title(f'd Offset Sampling — {len(D_OFFSETS)} candidates across the lane')
plt.tight_layout(); plt.show()
print(f'각 d 값마다 Reference와 평행한 후보 경로 1개씩 -> 총 {len(D_OFFSETS)}개')


## 2. 후보 경로 생성 + 충돌 검사

### 이론 배경

**생성**: 법선 방향으로 d만큼 이동  
$$x_{cand} = x_r - d\sin\theta_r, \quad y_{cand} = y_r + d\cos\theta_r$$

**충돌 검사**: 원형 영역 (point-in-circle)  
$$\text{충돌} \iff \min \|p - o\| < r_{robot}$$

**입력**: d 후보 + 장애물 / **출력**: 유효(초록) / 충돌(빨강) 분류

In [ ]:
valid_paths, invalid_paths = [], []
for d in D_OFFSETS:
    cx, cy = generate_candidate(rx, ry, ryaw, d)
    (valid_paths if is_collision_free(cx, cy, obs_x, obs_y, ROBOT_RADIUS)
     else invalid_paths).append((d, cx, cy))

print('충돌 검사 결과')
print(f'  유효: {len(valid_paths)}개  /  충돌: {len(invalid_paths)}개')

fig, ax = plt.subplots(figsize=(12, 7))
ax.plot(rx, ry, '-b', lw=3, alpha=0.4, label='Reference Path')
fi = fv = True
for d, cx, cy in invalid_paths:
    ax.plot(cx, cy, '-r', lw=1.5, alpha=0.7, label='Collision' if fi else ''); fi=False
for d, cx, cy in valid_paths:
    ax.plot(cx, cy, '-g', lw=1.5, alpha=0.7, label='Valid' if fv else ''); fv=False
for ox, oy in zip(obs_x, obs_y):
    ax.add_patch(plt.Circle((ox,oy), ROBOT_RADIUS, color='red', alpha=0.25))
    ax.plot(ox, oy, 'rx', ms=8, mew=2)
ax.set_aspect('equal'); ax.grid(True, alpha=0.3); ax.legend(loc='upper right', fontsize=9)
ax.set_title(f'Collision Check — Red: Collision({len(invalid_paths)}), Green: Valid({len(valid_paths)})')
plt.tight_layout(); plt.show()
print('유효 경로(초록)만 다음 비용 평가 단계로 전달')


## 3. 비용 함수 3종

### 이론 배경

| 비용 | 수식 | 의미 |
|------|------|------|
| Safety | $1/d_{min}$ | 장애물 가까울수록 큼 |
| Comfort | $\sum \Delta\theta^2$ | 방향 변화 클수록 큼 |
| Reference | $d^2$ | 중심선 이탈 클수록 큼 |

**입력**: 유효 경로 / **출력**: 각 경로의 3종 비용 — 세 비용의 최소 위치가 서로 다름에 주목.

In [ ]:
ds   = [d for d, *_ in valid_paths]
sft  = [cost_safety(cx,cy,obs_x,obs_y) for _,cx,cy in valid_paths]
cft  = [cost_comfort(cx,cy)            for _,cx,cy in valid_paths]
ref  = [cost_reference(d)               for d,*_ in valid_paths]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, vals, name, col in [(axes[0],sft,'Safety (1/min_dist)','tomato'),
                            (axes[1],cft,'Comfort (sum dyaw^2)','steelblue'),
                            (axes[2],ref,'Reference (d^2)','seagreen')]:
    ax.plot(ds, vals, 'o-', color=col, lw=2, ms=7)
    ax.plot(ds[int(np.argmin(vals))], min(vals), '*', color='black', ms=16, zorder=5)
    ax.set_xlabel('d offset [m]'); ax.set_title(name); ax.grid(True, alpha=0.3)
axes[0].set_ylabel('cost value')
plt.suptitle('Each Cost Component vs d (black star = minimum)', fontsize=12)
plt.tight_layout(); plt.show()
print('Safety: 장애물 먼 쪽 / Comfort: 직선 가까운 쪽 / Reference: d=0')
print('-> 세 비용의 최소 위치가 달라서 가중치로 절충해야 함')


## 4. 가중치 시나리오 비교

### 이론 배경

$$C_{total} = w_s C_{safety} + w_c C_{comfort} + w_r C_{ref}$$

비용들의 **크기 단위가 다르므로** 그대로 더하면 큰 비용이 가중치를 무력화합니다.  
→ 각 비용을 **[0,1]로 정규화**해야 가중치가 제대로 작동합니다.

**입력**: 비용 + 가중치 / **출력**: 시나리오별로 다르게 선택되는 최적 경로

In [ ]:
SCENARIOS = [
    ('Safety Priority',  dict(w_s=5.0, w_c=1.0, w_r=1.0)),
    ('Balanced',         dict(w_s=2.0, w_c=2.0, w_r=2.0)),
    ('Comfort Priority', dict(w_s=1.0, w_c=5.0, w_r=1.0)),
    ('Center Line',      dict(w_s=1.0, w_c=1.0, w_r=5.0)),
]
# 비용 크기 단위가 달라(safety~0.3, reference~9) 그대로 더하면 큰 값이 가중치를
# 무력화 -> 각 비용을 [0,1]로 정규화해야 가중치가 의미를 가짐.
ns = normalize([cost_safety(cx,cy,obs_x,obs_y) for _,cx,cy in valid_paths])
nc = normalize([cost_comfort(cx,cy)            for _,cx,cy in valid_paths])
nr = normalize([cost_reference(d)               for d,*_ in valid_paths])

print('시나리오별 최적 경로 (정규화 비용)')
print(f'  {"Scenario":<18}  {"w_s":>5} {"w_c":>5} {"w_r":>5}  {"Best d":>7}')
print('  ' + '-'*48)
bests = []
for name, w in SCENARIOS:
    costs = [w['w_s']*ns[i] + w['w_c']*nc[i] + w['w_r']*nr[i] for i in range(len(valid_paths))]
    best = valid_paths[int(np.argmin(costs))]
    bests.append((name, w, best))
    print(f"  {name:<18}  {w['w_s']:>5.1f} {w['w_c']:>5.1f} {w['w_r']:>5.1f}  {best[0]:>+7.1f}")
print('  -> 가중치에 따라 다른 경로 선택 (정규화 덕분)')

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, (name, w, (bd, bcx, bcy)) in zip(axes.flat, bests):
    ax.plot(rx, ry, '-b', lw=2, alpha=0.25)
    for d, cx, cy in valid_paths: ax.plot(cx, cy, '-', lw=1, alpha=0.3, color='gray')
    ax.plot(bcx, bcy, '-g', lw=2.5, label=f'Optimal d={bd:+.1f}m')
    for ox, oy in zip(obs_x, obs_y): ax.add_patch(plt.Circle((ox,oy), ROBOT_RADIUS, color='red', alpha=0.25))
    ax.plot(obs_x, obs_y, 'rx', ms=7, mew=2)
    ax.set_aspect('equal'); ax.grid(True, alpha=0.3); ax.legend(fontsize=8)
    ax.set_title(f'{name} -> d={bd:+.1f}m  (w_s={w["w_s"]},w_c={w["w_c"]},w_r={w["w_r"]})', fontsize=9)
plt.suptitle('Optimal Path by Weight Scenario (different weights -> different choice)', fontsize=13)
plt.tight_layout(); plt.show()


## 5. 최적 경로 선택 (통합 파이프라인)

### 이론 배경

충돌 없는 후보 → 비용 계산 → 정규화 → 가중 합산 → **최소 비용 선택**.  
실제 시스템에서는 이 사이클이 **10~20Hz**로 반복됩니다.

**입력**: 유효 경로 + 가중치 / **출력**: 단일 최적 경로 + 후보 랭킹

In [ ]:
W_SAFETY, W_COMFORT, W_REF = 3.0, 1.0, 1.0   # 안전 약간 우선

# 충돌 없는 후보 -> 원시 비용 -> 정규화 -> 가중 합산 -> 정렬
collected = []
for d, cx, cy in valid_paths:
    collected.append([d, cx, cy,
                      cost_safety(cx,cy,obs_x,obs_y), cost_comfort(cx,cy), cost_reference(d)])
gs_ = normalize([r[3] for r in collected])
gc_ = normalize([r[4] for r in collected])
gr_ = normalize([r[5] for r in collected])
candidates = sorted(
    [(W_SAFETY*gs_[i] + W_COMFORT*gc_[i] + W_REF*gr_[i], r[0], r[1], r[2])
     for i, r in enumerate(collected)], key=lambda t: t[0])
best_cost, best_d, best_cx, best_cy = candidates[0]

print(f'최적 경로 선택 (w_s={W_SAFETY}, w_c={W_COMFORT}, w_r={W_REF})')
print(f'  {"d (m)":>7}  {"비용":>9}  순위')
print('  ' + '-'*28)
for rank, (cost, d, cx, cy) in enumerate(candidates, 1):
    print(f'  {d:>+7.1f}  {cost:>9.4f}  {rank}{"  <- Optimal" if rank==1 else ""}')

fig, ax = plt.subplots(figsize=(12, 7))
ax.plot(rx, ry, '-b', lw=2, alpha=0.25, label='Reference Path')
for cost, d, cx, cy in candidates[1:]:
    ax.plot(cx, cy, '-', lw=1, alpha=0.35, color='gray')
ax.plot(best_cx, best_cy, '-g', lw=3, label=f'Optimal Path (d={best_d:+.1f}m)')
for ox, oy in zip(obs_x, obs_y): ax.add_patch(plt.Circle((ox,oy), ROBOT_RADIUS, color='red', alpha=0.25))
ax.plot(obs_x, obs_y, 'rx', ms=8, mew=2, label='Obstacle')
ax.plot(rx[0], ry[0], 'og', ms=10, label='Start'); ax.plot(rx[-1], ry[-1], 'sb', ms=10, label='Goal')
ax.set_aspect('equal'); ax.grid(True, alpha=0.3); ax.legend(loc='upper right', fontsize=8)
ax.set_title('Optimal Path Selection Result')
plt.tight_layout(); plt.show()


## 6. Pure Pursuit 추종

### 이론 배경

선택된 경로를 차량이 따라갑니다. **Pure Pursuit**는 `look_ahead` 앞 목표점을 향해 조향:

$$\delta = \arctan\!\left(\frac{2\,WB\,\sin\alpha}{L_{ld}}\right)$$

**입력**: 최적 경로 / **출력**: 차량 주행 궤적 애니메이션

In [ ]:
path_x, path_y = best_cx, best_cy
speed, look_ahead, dt = 10.0, 8.0, 0.1
step = speed * dt

x, y = path_x[0], path_y[0]
yaw = math.atan2(path_y[1]-path_y[0], path_x[1]-path_x[0])
rem_x, rem_y = list(path_x), list(path_y)
hist_x, hist_y, frames = [x], [y], []
for _ in range(3000):
    if len(rem_x) < 2: break
    steer, idx = pure_pursuit_steer(x, y, yaw, rem_x, rem_y, look_ahead)
    x, y, yaw = move(x, y, yaw, step, steer)
    rem_x, rem_y = rem_x[max(0,idx-1):], rem_y[max(0,idx-1):]
    hist_x.append(x); hist_y.append(y)
    frames.append((x, y, yaw, list(hist_x), list(hist_y)))
    if math.hypot(path_x[-1]-x, path_y[-1]-y) < 2.0: break

fig, ax = plt.subplots(figsize=(12, 7))
ax.plot(rx, ry, '--', color='lightblue', lw=1, label='Reference Path')
ax.plot(path_x, path_y, '-g', lw=2, label='Optimal Path')
for ox, oy in zip(obs_x, obs_y): ax.add_patch(plt.Circle((ox,oy), ROBOT_RADIUS, color='red', alpha=0.25))
ax.set_aspect('equal'); ax.grid(True, alpha=0.3); ax.legend(loc='upper right', fontsize=8)
ax.set_title('Vehicle Tracking — Pure Pursuit Controller')
trail, = ax.plot([], [], '-b', lw=1.5, alpha=0.7)
car_lines = []
def update_vehicle(fd):
    global car_lines
    for ln in car_lines: ln.remove()
    car_lines = []
    fx, fy, fyaw, hx, hy = fd
    trail.set_data(hx, hy)
    c, s = math.cos(fyaw), math.sin(fyaw)
    pts = np.array([[c,-s],[s,c]]) @ np.array([VRX, VRY])
    ln, = ax.plot(pts[0]+fx, pts[1]+fy, color='royalblue', lw=1.5)
    car_lines.append(ln)
    return trail,
sampled = frames[::max(1, len(frames)//120)]
ani_track = FuncAnimation(fig, update_vehicle, frames=sampled, interval=50, repeat=False, blit=False)
HTML(ani_track.to_jshtml())


## 7. 정리 및 한계

### 이론 배경

이 Local Planner는 **정적 평행 오프셋 후보**를 한 번 평가해 선택합니다.  
직관적이지만 시간·속도·차량 동역학을 고려하지 못합니다.  
→ 다음 **Step 05 Frenet Optimal Trajectory**: Quintic 다항식 시간 궤적 + 동역학 제약 분류 + receding horizon 재계획.

In [ ]:
print('Frenet 기반 Local Planner 완성!')
print()
print('이 노트북에서 통합한 단계:')
for s, t in [('1','d 오프셋 샘플링'),('2','후보 생성 + 충돌 검사'),
             ('3','비용 3종 (Safety/Comfort/Reference)'),('4','가중치 시나리오 비교'),
             ('5','정규화 + 최적 경로 선택'),('6','Pure Pursuit 추종')]:
    print(f'  단계 {s}: {t}')
print()
print('현재 구현의 한계 (-> 다음 Step 05 Frenet Optimal Trajectory에서 개선):')
for lim in ['정적 후보(평행 오프셋)만 사용 — 시간/속도 미고려',
            'd 이산 샘플만 탐색 — 연속 최적해 누락 가능',
            '한 번 선택 후 끝 — 실제론 매 스텝 재계획(receding horizon) 필요',
            '차량 동역학(속도/가속/jerk) 제약 미반영']:
    print(f'  - {lim}')
